In [ ]:
# checks that all observations - draw-time average is near equal for all observations
# should return 0
def observation_ave_close(post1, post2,n):
    diff_ge_1 = []
    cnt = 0
    for i in range(n):
        p1 = float(post1.isel(p_obs=i).mean())
        p2 = float(post2["mu"].isel(p_obs=i).mean())
        diff= p1-p2
        if abs(diff) > .001:
            diff_ge_1.append((i,p1,p2,diff))
            cnt+=1
    print(cnt)
    return diff_ge_1, cnt

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Calculate the expected value (mean probability) for EVERY patient
# (Replace 'post1' with whatever your PyMC native output variable is named)
py_patient_means = post1.mean(dim=["sample"]).values  # Native PyMC
nb_patient_means = post2["mu"].mean(dim=["chain", "draw"]).values  # Numba Engine

# 2. Plot them against each other
plt.figure(figsize=(8, 8))
plt.scatter(py_patient_means, nb_patient_means, alpha=0.4, color='blue', edgecolor='none')

# Draw the perfect 1:1 match line
max_val = max(np.max(py_patient_means), np.max(nb_patient_means))
plt.plot([0, max_val], [0, max_val], 'k--', lw=2, label="Perfect Match (y=x)")

plt.xlabel("PyMC Native Expected Probability")
plt.ylabel("Numba Engine Expected Probability")
plt.title("Per-Patient Verification: PyMC vs Numba")
plt.legend()
plt.grid(True)
plt.show()

# Calculate the mean absolute error across the population
mae = np.mean(np.abs(py_patient_means - nb_patient_means))
print(f"Mean Absolute Error across {len(py_patient_means)} patients: {mae:.6f}")